# Chunking and article aggregation evaluation

Compare the three BGE-M3 recursive chunking variants and article score aggregation methods using the same legal relevance mapping.

In [ ]:
import json
import math
import unicodedata
from pathlib import Path

import numpy as np
from sentence_transformers import SentenceTransformer

PROJECT_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'data-pipeline').is_dir())
DATA_DIR = PROJECT_ROOT / 'data-pipeline' / 'data'
PROCESSED_DIR = DATA_DIR / 'processed'
EMBEDDING_DIR = PROCESSED_DIR / 'test' / 'embeddings'
EVALUATION_DIR = PROJECT_ROOT / 'data-pipeline' / 'evaluation' / 'data'
VARIANTS = ['recursive_300_chars', 'recursive_450_tokens', 'recursive_600_tokens']
AGGREGATIONS = ['best', 'mean_top_2', 'mean_top_3']

def load_jsonl(path):
    return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]

def canonical(article):
    return str(article.get('article_id') or f"{article.get('document_id')}-{article.get('article_number')}")

articles = {}
for article in load_jsonl(PROCESSED_DIR / 'articles_normalized.jsonl'):
    articles.setdefault(canonical(article), article)
article_ids = list(articles)
article_vectors = np.load(EMBEDDING_DIR / 'articles_bge_m3.npy')
assert len(article_ids) == len(article_vectors), (len(article_ids), len(article_vectors))
mapping = json.loads((EVALUATION_DIR / 'relevance_mapping.json').read_text(encoding='utf-8'))
qrels = {item['query_id']: {article_id: 2 for article_id in item['relevant_articles']} for item in mapping}
queries = {item['query_id']: item['query'] for item in mapping}
eligible = {qid: labels for qid, labels in qrels.items() if set(labels) & set(article_ids)}
model = SentenceTransformer('BAAI/bge-m3')

def ndcg(ranked, labels, k=3):
    gains = [(2 ** labels.get(article_id, 0)) - 1 for article_id in ranked[:k]]
    dcg = sum(gain / math.log2(index + 2) for index, gain in enumerate(gains))
    ideal = sorted(((2 ** value) - 1 for value in labels.values()), reverse=True)[:k]
    idcg = sum(gain / math.log2(index + 2) for index, gain in enumerate(ideal))
    return dcg / idcg if idcg else 0.0

def metrics(ranked, labels):
    relevant = set(labels)
    hits = [index + 1 for index, article_id in enumerate(ranked) if article_id in relevant]
    rank = hits[0] if hits else None
    return {'hit@3': int(rank is not None and rank <= 3), 'hit@5': int(rank is not None and rank <= 5), 'mrr@3': 1 / rank if rank and rank <= 3 else 0.0, 'ndcg@3': ndcg(ranked, labels), 'rank': rank}

def summarize(rows):
    return {key: round(sum(row[key] for row in rows) / len(rows), 4) for key in ('hit@3', 'hit@5', 'mrr@3', 'ndcg@3')}

report = {'query_count': len(qrels), 'eligible_query_count': len(eligible), 'excluded_query_ids': sorted(set(qrels) - set(eligible)), 'variants': {}}
for variant in VARIANTS:
    prefix = f'chunks_bge_m3_{variant}'
    records = load_jsonl(EMBEDDING_DIR / f'{prefix}_metadata.jsonl')
    vectors = np.load(EMBEDDING_DIR / f'{prefix}.npy')
    variant_rows = {aggregation: [] for aggregation in AGGREGATIONS}
    for query_id, labels in eligible.items():
        query = unicodedata.normalize('NFC', queries[query_id]).replace('\u200b', '').replace('\u200c', '').replace('\u200d', '')
        query_vector = model.encode([query], convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False)[0]
        scores = vectors @ query_vector
        grouped = {}
        for index in np.argsort(-scores): grouped.setdefault(str(records[index]['article_id']), []).append(float(scores[index]))
        for aggregation in AGGREGATIONS:
            ranked = sorted(grouped, key=lambda article_id: (sum(sorted(grouped[article_id], reverse=True)[:int(aggregation[-1])] if aggregation.startswith('mean_top_') else [max(grouped[article_id])]) / (int(aggregation[-1]) if aggregation.startswith('mean_top_') else 1)), reverse=True)
            variant_rows[aggregation].append({'query_id': query_id, **metrics(ranked, labels)})
    report['variants'][variant] = {aggregation: {'metrics': summarize(rows), 'results': rows} for aggregation, rows in variant_rows.items()}

output = PROCESSED_DIR / 'test' / 'reports' / 'chunking_and_aggregation_evaluation.json'
output.parent.mkdir(parents=True, exist_ok=True)
output.write_text(json.dumps(report, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print(json.dumps({'query_count': report['query_count'], 'eligible_query_count': report['eligible_query_count'], 'variants': {name: {agg: value['metrics'] for agg, value in data.items()} for name, data in report['variants'].items()}}, ensure_ascii=False, indent=2))
print('Saved:', output)